# Поиск фильма через OMDb API

Выберите ядро из окружения `.venv` и выполните ячейки сверху вниз. Ноутбук отправляет запрос по названию фильма и показывает данные из JSON-ответа.

In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

## Название фильма и ключ из окружения

Скопируйте `.env.example` в `.env` рядом с ноутбуком и укажите свой ключ в `OMDB_API_KEY`. Запустите ноутбук из папки проекта. Если вы изменили ключ, перезапустите ядро и запустите ячейки заново.

In [ ]:
load_dotenv()
api_key = os.getenv("OMDB_API_KEY", "").strip()
if api_key:
    title = input("Введите название фильма: ").strip()
else:
    print("Ключ OMDB_API_KEY отсутствует. Укажите его в локальном файле .env.")

## Запрос и полный JSON

`requests` отправляет GET-запрос с параметрами `t` (название) и `apikey` (ключ). Проверяем HTTP-статус и поле `Response`. При успешном поиске выводим полный JSON с отступами.

In [ ]:
def get_movie(title, api_key):
    try:
        response = requests.get(
            "https://www.omdbapi.com/",
            params={"t": title, "apikey": api_key},
            timeout=10,
        )
        response.raise_for_status()
    except requests.HTTPError as error:
        status = error.response.status_code
        if status in (401, 403):
            print("Доступ к OMDb отклонён. Проверьте ключ OMDb.")
        else:
            print(f"OMDb вернул ошибку HTTP {status}.")
        return None
    except requests.RequestException:
        print("Не удалось связаться с OMDb.")
        return None

    try:
        data = response.json()
    except ValueError:
        print("OMDb вернул ответ, который не удалось прочитать как JSON.")
        return None

    if not isinstance(data, dict):
        print("В ответе OMDb ожидался объект JSON.")
        return None

    if data.get("Response") == "False":
        reason = data.get("Error")
        if reason == "Movie not found!":
            print("Фильм не найден. Проверьте название.")
        elif reason == "Invalid API key!":
            print("Доступ к OMDb отклонён. Проверьте ключ OMDb.")
        else:
            print("OMDb отказал в обработке запроса.")
        return None

    return data


movie = None
if api_key:
    movie = get_movie(title, api_key)
del api_key
if movie is not None:
    print(json.dumps(movie, ensure_ascii=False, indent=2))

## Данные о фильме

Сводка показывает главные поля, включая актёров. Метод `.get()` позволяет указать значение на случай, если нужного поля нет.

In [ ]:
if movie is not None:
    print("\n=== Результат поиска ===")
    print(f"Название: {movie.get('Title', 'нет данных')}")
    print(f"Год: {movie.get('Year', 'нет данных')}")
    print(f"Жанр: {movie.get('Genre', 'нет данных')}")
    print(f"Режиссёр: {movie.get('Director', 'нет данных')}")
    print(f"Рейтинг IMDb: {movie.get('imdbRating', 'нет данных')}")
    print(f"Актёры: {movie.get('Actors', 'нет данных')}")